# **Perkenalan Dataset**

**Tentang Data**

---

*   Sumber Data kaggle.com
*   Sumber Data “Bank Transaction Dataset for Fraud Detection”
*.  Modif by Dicoding
*   Oleh Maziya Ats Tsaqofi
*   Tanggal 30 Mei 2026

# **1. Import Library**

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# **2. Memuat Dataset Hasil Clustering**

In [6]:
# Connect to Drive
from google.colab import drive
drive.mount('/content/drive')

# Memuat Dataset dari drive
file_path = '/content/drive/My Drive/Colab Notebooks/DicodingProject/data_clustering.csv'
df = pd.read_csv(file_path)

df.head()
df.info()
df.columns.tolist()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2135 entries, 0 to 2134
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   TransactionAmount    2135 non-null   float64
 1   TransactionType      2135 non-null   int64  
 2   Location             2135 non-null   int64  
 3   Channel              2135 non-null   int64  
 4   CustomerAge          2135 non-null   float64
 5   CustomerOccupation   2135 non-null   int64  
 6   TransactionDuration  2135 non-null   float64
 7   LoginAttempts        2135 non-null   float64
 8   AccountBalance       2135 non-null   float64
 9   Age_Bin              2135 non-null   int64  
 10  Target               2135 non-null   int64  
dtypes: float64(5), int64(6)
memory usage: 183.6 KB


['TransactionAmount',
 'TransactionType',
 'Location',
 'Channel',
 'CustomerAge',
 'CustomerOccupation',
 'TransactionDuration',
 'LoginAttempts',
 'AccountBalance',
 'Age_Bin',
 'Target']

In [7]:
X = df.drop('Target', axis=1)
print(X.shape)

y = df['Target']
print(y.shape)

(2135, 10)
(2135,)


# **3. Data Spliting**

In [10]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1708, 10)
(427, 10)
(1708,)
(427,)


# **4. Membangun Model Klasifikasi**

In [12]:
# Menjalankan decision tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

In [14]:
# Evaluasi model
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, average='weighted'))
print("Recall:", recall_score(y_test, y_pred_dt, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred_dt, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.9742388758782201
Precision: 0.9743478243974217
Recall: 0.9742388758782201
F1-Score: 0.974152966339811

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.95      0.96      0.96       127
           2       0.97      1.00      0.99       110
           3       0.98      0.94      0.96       104
           4       1.00      1.00      1.00        70

    accuracy                           0.97       427
   macro avg       0.98      0.98      0.98       427
weighted avg       0.97      0.97      0.97       427



In [15]:
joblib.dump(dt_model, "decision_tree_model.h5")

['decision_tree_model.h5']

# **5. Memenuhi kriteria Skilled**

In [17]:
rf_model = RandomForestClassifier(
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [18]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, average='weighted'))
print("Recall:", recall_score(y_test, y_pred_rf, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred_rf, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.9882903981264637
Precision: 0.9883204652596528
Recall: 0.9882903981264637
F1-Score: 0.9882731824104903

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.98      0.98      0.98       127
           2       0.99      1.00      1.00       110
           3       0.99      0.97      0.98       104
           4       1.00      1.00      1.00        70

    accuracy                           0.99       427
   macro avg       0.99      0.99      0.99       427
weighted avg       0.99      0.99      0.99       427



In [19]:
joblib.dump(rf_model, "explore_RandomForest_classification.h5")

['explore_RandomForest_classification.h5']

In [21]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100]},
             scoring='f1_weighted')

In [22]:
best_model = grid_search.best_estimator_

y_pred_tuning = best_model.predict(X_test)

print("Best Parameter:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_tuning))
print("Precision:", precision_score(y_test, y_pred_tuning, average='weighted'))
print("Recall:", recall_score(y_test, y_pred_tuning, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred_tuning, average='weighted'))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuning))

Best Parameter: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Accuracy: 0.9859484777517564
Precision: 0.9859241019514775
Recall: 0.9859484777517564
F1-Score: 0.9859254921516462

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.98      0.98      0.98       127
           2       0.99      1.00      1.00       110
           3       0.98      0.97      0.98       104
           4       1.00      1.00      1.00        70

    accuracy                           0.99       427
   macro avg       0.99      0.99      0.99       427
weighted avg       0.99      0.99      0.99       427



In [23]:
joblib.dump(best_model, "tuning_classification.h5")

['tuning_classification.h5']

# **Noted**

### Metode yang Digunakan
Model klasifikasi dibangun menggunakan Decision Tree, Random Forest, dan Random Forest dengan hyperparameter tuning menggunakan GridSearchCV.

### Alasan Penggunaan
Decision Tree digunakan karena menjadi model wajib pada kriteria submission. Random Forest digunakan sebagai model pembanding karena mampu meningkatkan stabilitas prediksi dengan menggabungkan banyak pohon keputusan. GridSearchCV digunakan untuk mencari kombinasi parameter terbaik.

### Hasil yang Didapat
Decision Tree menghasilkan F1-Score sebesar 0.9741, Random Forest menghasilkan F1-Score sebesar 0.9882, dan model tuning menghasilkan F1-Score sebesar 0.9859. Berdasarkan hasil evaluasi, Random Forest tanpa tuning memberikan performa terbaik pada data testing.